Podešavanja

In [4]:
DATA_DIR = r"raw-img"
SEED = 42
MAX_PER_CLASS = 300

In [2]:
import os
import sys
from PIL import Image
import warnings
import torch
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
from torchvision import transforms
from pathlib import Path
import torch.nn as nn

sys.path.insert(0, str('src'))
from dataset import (
    Animals10Dataset,
    CLASSES, NUM_CLASSES, IMG_SIZE,
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
PAL = sns.color_palette('tab10', n_colors=10)

DEVICE = torch.device('cpu')

print(f'PyTorch   : {torch.__version__}')
print(f'MLflow    : {mlflow.__version__}')
print(f'Uredjaj   : {DEVICE}')
print(f'Klase     : {CLASSES}')

PyTorch   : 2.12.0+cpu
MLflow    : 3.13.0
Uredjaj   : cpu
Klase     : ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']


Fiksiranje sidova

In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seed fiksiran: {seed}")

seed_everything(SEED)

Seed fiksiran: 42


Mapa klasa i transformacije

In [3]:
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225],
)

def get_transform(split='train', strong=False):
    if split == 'train' and not strong:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            NORMALIZE,
        ])
    elif split == 'train' and strong:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.1),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.2),
            transforms.RandomGrayscale(p=0.05),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            NORMALIZE,
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            NORMALIZE,
        ])

print('Transformacije definisane!')
print(f'IMG_SIZE  = {IMG_SIZE}px')
print(f'Normalizacija: mean={[0.485, 0.456, 0.406]}  std={[0.229, 0.224, 0.225]}')

Transformacije definisane!
IMG_SIZE  = 128px
Normalizacija: mean=[0.485, 0.456, 0.406]  std=[0.229, 0.224, 0.225]


Učitavanje dataseta

In [6]:
Path('results').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)


full_ds = Animals10Dataset(
    root_dir=DATA_DIR,
    transform=None,
    max_per_class=MAX_PER_CLASS,
)

if len(full_ds) == 0:
    print('\n!!! GRESKA: Nisu pronadjene slike!')
   
else:
    all_labels = full_ds.get_labels()
    counts     = full_ds.class_counts()
    print(f'Klase i broj slika:')
    for cls, cnt in counts.items():
        print(f'  {cls:<12}: {cnt}')

Dataset ucitan: 3000 slika, 10 klasa
Klase i broj slika:
  butterfly   : 300
  cat         : 300
  chicken     : 300
  cow         : 300
  dog         : 300
  elephant    : 300
  horse       : 300
  sheep       : 300
  spider      : 300
  squirrel    : 300


Definisanje CNN arhitektura

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128*4*4, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

class MediumCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()
        def blok(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                )
        self.net = nn.Sequential(
            blok(3, 32), blok(32, 64), blok(64, 128), blok(128, 256),
            nn.AdaptiveAvgPool2d((3, 3)),
            nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(256*3*3, 512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes),
            )   
    def forward(self, x): return self.net(x)

class DeepCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        cfg    = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M']
        layers = []
        ic = 3
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(2))
            else:
                layers += [nn.Conv2d(ic, v, 3, padding=1), nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                ic = v
        self.features    = nn.Sequential(*layers)
        self.gap         = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.gap(self.features(x)))    
    
class TransferMobileNet(nn.Module):
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()
        from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
        backbone        = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        self.features   = backbone.features
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(1280, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.pool(self.features(x)))